# 第 22 章互动实验：能不能看见这个缺陷

用你的成像仿真器回答一个设计问题：在给定光学和曝光下，多大的缺陷能被检出。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "22_成像与设备仿真"
ASSIGNMENT = "22-1_成像仿真.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
for lam, na in [(532, 0.5), (532, 0.9), (266, 0.9), (193, 0.95)]:
    r = attempt(mod.rayleigh_resolution_um, lam, na)
    if r is None: break
    print(f"波长 {lam} nm, NA {na}：分辨率 {r:.3f} um，建议像素（物面）不大于 {r / 2:.3f} um")

## 实验 1：缺陷信噪比 vs 缺陷尺寸

场景：反射率 0.5 的平坦背景上一个亮颗粒（比背景高 0.3）。PSF sigma 固定，改变颗粒直径和曝光量。

- 颗粒按**覆盖比例**画：把每个像素切成 16×16 个小格，像素的值取落在圆里的小格所占的比例（`disk_coverage`）。如果只看像素中心在不在圆里，直径 0.5、1、1.5 像素都只点亮同一个像素，曲线的前三个点会完全相同（教程第 12.1 节）。
- 曝光量用“反射率 1 对应的平均电子数”表示（`exposure_e`）。作业的模型里没有满阱这个参数，饱和只发生在 ADU 超过 `2**bit_depth - 1` 的时候。这里是 12 位、增益 2 e-/ADU，上限 4095 ADU 对应 8190 e-，下面三档曝光都没有到。

In [ ]:
def disk_coverage(diam_px, size=41, sub=16):
    """圆心在正中那个像素的中心、直径 diam_px（像素）的圆：返回每个像素被圆盖住的比例，0 到 1。"""
    c = (size - 1) / 2
    t = (np.arange(size * sub) + 0.5) / sub - 0.5          # 每个小格中心的坐标，单位是像素
    inside = (t[:, None] - c) ** 2 + (t[None, :] - c) ** 2 <= (diam_px / 2) ** 2
    return inside.reshape(size, sub, size, sub).mean(axis=(1, 3))

def defect_snr(diam_px, exposure_e, psf_sigma=1.5, contrast=0.3, trials=30):
    scene = 0.5 + contrast * disk_coverage(diam_px)
    k = mod.gaussian_psf(psf_sigma)
    sharp = mod.convolve_same(scene, k)
    bg = mod.convolve_same(np.full_like(scene, 0.5), k)
    rng = np.random.default_rng(0)
    diffs = []
    for _ in range(trials):
        a = mod.simulate_capture(sharp * exposure_e, 5.0, 2.0, 12, rng).astype(float)
        b = mod.simulate_capture(bg * exposure_e, 5.0, 2.0, 12, rng).astype(float)
        diffs.append((a - b)[20, 20])
    noise = np.std([mod.simulate_capture(bg * exposure_e, 5.0, 2.0, 12, rng)[20, 20] for _ in range(200)]) * np.sqrt(2)
    return np.mean(diffs) / noise

diams = [0.5, 1, 1.5, 2, 3, 4, 6]
print("各直径的覆盖面积（像素）：", [round(float(disk_coverage(d).sum()), 2) for d in diams])
for exposure_e in (500, 2000, 6000):
    snr = attempt(lambda: [defect_snr(d, exposure_e) for d in diams])
    if snr is None: break
    plt.plot(diams, snr, "o-", label=f"反射率 1 对应 {exposure_e} e-")
else:
    plt.axhline(5, color="k", ls="--", label="SNR=5 参考线")
    plt.xlabel("缺陷直径 (px)"); plt.ylabel("中心像素信噪比"); plt.legend(); plt.show()

## 实验 2：光子转移曲线

In [ ]:
rng = np.random.default_rng(3)
levels = np.array([20, 50, 100, 300, 800, 1500, 3000, 6000, 12000, 20000], float)
means, vars_ = [], []
for lv in levels:
    f = attempt(mod.simulate_capture, np.full((100, 100), lv), 8.0, 1.5, 12, rng)
    if f is None: break
    means.append(f.mean()); vars_.append(f.var())
if len(means) == len(levels):
    plt.loglog(means, vars_, "o-"); plt.xlabel("均值 (ADU)"); plt.ylabel("方差 (ADU²)"); plt.title("PTC：注意高端的饱和弯折"); plt.show()
    ok = np.array(means) < 0.7 * 4095
    est = attempt(mod.estimate_gain_from_ptc, np.array(means)[ok], np.array(vars_)[ok])
    if est is not None:
        print("只用未饱和点拟合：", est, "真值 (1.5, 8.0)")

## 实验 3：暗处的统计量和黑电平

读出噪声有正有负。信号很暗时，电子数里负的那一部分换成 ADU 之后被裁成 0，均值被抬高，标准差被压小。`simulate_capture` 的 `black_level_adu` 在量化之前加一个固定偏置，把整个分布抬离 0。

下面拍全暗的一帧，读出噪声 8 e-、增益 1.5 e-/ADU，比较不加黑电平和加 64 ADU 两种情况。先预测：标准差应该是多少 ADU？不加黑电平时会偏大还是偏小？

In [ ]:
rng = np.random.default_rng(5)
read_e, gain = 8.0, 1.5
want = np.sqrt((read_e / gain) ** 2 + 1 / 12)          # 读出噪声换算成 ADU，再加四舍五入带来的量化噪声
frames = {}
for black in (0.0, 64.0):
    f = attempt(mod.simulate_capture, np.zeros((300, 300)), read_e, gain, 12, rng, black_level_adu=black)
    if f is None: break
    frames[black] = f
    print(f"黑电平 {black:2.0f} ADU：均值减去黑电平 {f.mean() - black:5.2f} ADU，标准差 {f.std():.2f} ADU（没有裁剪时应约 {want:.2f}），等于 0 的像素占 {np.mean(f == 0):.0%}")
if len(frames) == 2:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
    for a, (black, f) in zip(ax, frames.items()):
        a.hist(f.ravel() - black, bins=np.arange(-25.5, 26.5)); a.set_xlabel("像素值减去黑电平 (ADU)"); a.set_title(f"黑电平 {black:.0f} ADU")
    plt.show()

## 破坏实验与记录

- 用全部点（含饱和点）拟合 PTC，增益估计偏向哪边？
- 把 PSF sigma 从 1.5 改成 3（离焦），实验 1 的曲线怎样？
- 在实验 2 的亮度档里加一档全暗（0 e-），分别不加黑电平和加 64 ADU 做拟合（加了黑电平要先从均值里减掉它），读出噪声的估计差多少？对照教程第 6.2 节。
- 写下你的结论：在你设定的参数下最小可检出缺陷多大，依据是什么。